In [3]:
from pathlib import Path
import csv
import json
import shutil

# --- 1. STUDENT CONFIGURATION ---
LAST_NAME = "Juarez"
STUDENT_ID = "TUPM-26-1843"
SEED_DIGIT = int(STUDENT_ID[-1])  # 3
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 24

# Directories setup
WORK_DIR = Path(".")
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

# --- 2. EXERCISE 2 RULES ---
target_os = "iOS"  # Since seed digit is odd (3)
target_behavior_class = (SEED_DIGIT % 5) + 1  # 4

device_dataset_path = Path("device_usage.csv")  # Ensure the dataset filename matches your workspace
device_filtered_csv = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_device_filtered.csv"
device_json_summary = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_device_summary.json"

# Required exported fields
exported_fields = [
    "User ID", 
    "Device Model", 
    "Operating System", 
    "Battery Drain (mAh/day)", 
    "Data Usage (MB/day)"
]

total_battery = 0.0
total_data = 0.0
matched_users = 0
device_rows = []

if device_dataset_path.exists():
    with open(device_dataset_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        
        for row in reader:
            os_val = row.get("Operating System", "").strip()
            behavior_val = row.get("User Behavior Class", "").strip()
            
            # Match the target OS and Behavior Class
            if os_val == target_os and behavior_val == str(target_behavior_class):
                # Extract only the specified exported fields
                filtered_row = {field: row.get(field, "") for field in exported_fields}
                device_rows.append(filtered_row)
                matched_users += 1
                
                try:
                    total_battery += float(row.get("Battery Drain (mAh/day)", 0))
                    total_data += float(row.get("Data Usage (MB/day)", 0))
                except ValueError:
                    pass

# Compute averages
avg_battery = total_battery / matched_users if matched_users else 0.0
avg_data = total_data / matched_users if matched_users else 0.0

# Export to Filtered CSV
if device_rows:
    with open(device_filtered_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=exported_fields)
        writer.writeheader()
        writer.writerows(device_rows)

# JSON Summary and Row Count Verification
device_summary = {
    "target_os": target_os,
    "target_behavior_class": target_behavior_class,
    "matched_records_count": matched_users,
    "exported_row_count": len(device_rows),
    "average_battery_drain_mAh_day": round(avg_battery, 2),
    "average_data_usage_MB_day": round(avg_data, 2)
}

with open(device_json_summary, "w", encoding="utf-8") as f:
    json.dump(device_summary, f, indent=4)

# CSV Backup and File Size Reporting
device_backup_csv = BACKUP_DIR / device_filtered_csv.name
if device_filtered_csv.exists():
    shutil.copy2(device_filtered_csv, device_backup_csv)
    file_size_bytes = device_filtered_csv.stat().st_size
    print(f"Backup created successfully: {device_backup_csv}")
    print(f"Exported CSV File Size: {file_size_bytes} bytes")

print(f"--- Exercise 2 Complete ---")
print(f"Total Matched Records: {matched_users}")
print(f"Average Battery Drain: {round(avg_battery, 2)} mAh/day")
print(f"Average Data Usage: {round(avg_data, 2)} MB/day")

--- Exercise 2 Complete ---
Total Matched Records: 0
Average Battery Drain: 0.0 mAh/day
Average Data Usage: 0.0 MB/day
